# Generate Complete Nutrition Database for 529 Ingredients

**Purpose**: Extract USDA nutrition data for all 529 ingredients in `ingredients_vocabulary.csv`

**Input**: 
- `data/ingredients_vocabulary.csv` (529 ingredients)
- `data/raw/nutrition_database/usda_nutrition_database.parquet` (USDA data)

**Output**:
- `data/ingredients_nutrition_full.csv` - Complete nutrition data for all 529 ingredients
- `data/nutrition_lookup_full.json` - JSON lookup table

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from tqdm import tqdm
import re

print("✓ Packages imported")

✓ Packages imported


In [2]:
# Paths
PROJECT_ROOT = Path.cwd().parent.parent
DATA_DIR = PROJECT_ROOT / "data"
DATA_RAW = DATA_DIR / "raw" / "nutrition_database"

# Input files
INGREDIENTS_CSV = DATA_DIR / "ingredients_vocabulary.csv"
USDA_DB = DATA_RAW / "usda_nutrition_database.parquet"

# Output files
OUTPUT_CSV = DATA_DIR / "ingredients_nutrition_full.csv"
OUTPUT_JSON = DATA_DIR / "nutrition_lookup_full.json"

print(f"Input:")
print(f"  - Ingredients: {INGREDIENTS_CSV}")
print(f"  - USDA DB: {USDA_DB}")
print(f"\nOutput:")
print(f"  - CSV: {OUTPUT_CSV}")
print(f"  - JSON: {OUTPUT_JSON}")

Input:
  - Ingredients: c:\Users\Champion\Documents\GitHub\cAIuldron\data\ingredients_vocabulary.csv
  - USDA DB: c:\Users\Champion\Documents\GitHub\cAIuldron\data\raw\nutrition_database\usda_nutrition_database.parquet

Output:
  - CSV: c:\Users\Champion\Documents\GitHub\cAIuldron\data\ingredients_nutrition_full.csv
  - JSON: c:\Users\Champion\Documents\GitHub\cAIuldron\data\nutrition_lookup_full.json


## 2. Load Data

In [3]:
# Load ingredients list
df_ingredients = pd.read_csv(INGREDIENTS_CSV)
ingredients_list = df_ingredients['Ingredient'].tolist()

print(f"✓ Loaded {len(ingredients_list)} ingredients")
print(f"\nFirst 10:")
for i, ing in enumerate(ingredients_list[:10], 1):
    print(f"  {i:2d}. {ing}")

✓ Loaded 528 ingredients

First 10:
   1. Chicken breast
   2. Chicken thigh
   3. Chicken drumstick
   4. Chicken wing
   5. Chicken liver
   6. Chicken gizzard
   7. Chicken feet
   8. Ground chicken
   9. Turkey breast
  10. Turkey thigh


In [4]:
# Load USDA database
if not USDA_DB.exists():
    print(f"⚠ USDA database not found: {USDA_DB}")
    print(f"\nPlease run 'load_usda_database.ipynb' first to download USDA data")
else:
    df_usda = pd.read_parquet(USDA_DB)
    print(f"✓ Loaded USDA database: {len(df_usda):,} food items")
    print(f"\nColumns: {list(df_usda.columns)}")
    print(f"\nSample data:")
    display(df_usda.head())

✓ Loaded USDA database: 1,858,059 food items

Columns: ['fdc_id', 'description', 'data_type', 'calories', 'carbs_g', 'fat_g', 'protein_g']

Sample data:


,fdc_id,description,data_type,calories,carbs_g,fat_g,protein_g
0,2571794,ALL NATURAL GLUTEN FREE CHICKEN NUGGETS,branded_food,194.0,8.24,9.41,18.82
1,2533357,ALL NATURAL GLUTEN FREE CHICKEN NUGGETS,branded_food,194.0,8.24,9.41,18.82
2,2617100,ALL NATURAL GLUTEN FREE CHICKEN NUGGETS,branded_food,194.0,8.24,9.41,18.82
3,2604504,ALL NATURAL ROSEMARY & OLIVE OIL BASMATI RICE...,branded_food,326.0,78.26,0.00,6.52
4,2583001,ALL NATURAL ROSEMARY & OLIVE OIL BASMATI RICE...,branded_food,326.0,78.26,0.00,6.52


## 3. Search Strategy

For each ingredient, we search USDA database using:
1. **Exact match** (e.g., "Chicken breast")
2. **Keywords** (e.g., "chicken" + "breast")
3. **Fallback to base ingredient** (e.g., "Chicken breast" → "chicken")

We prioritize:
- **SR Legacy Food** (Standard Reference - most accurate for raw ingredients)
- **Foundation Food** (Detailed nutrient profiles)
- **Survey FNDDS** (Common foods in US diet)
- **Branded Food** (last resort)

In [5]:
def clean_ingredient_name(name):
    """Clean ingredient name for better matching"""
    # Convert to lowercase
    cleaned = name.lower()
    # Remove common words that don't help matching
    remove_words = ['raw', 'fresh', 'organic', 'natural']
    for word in remove_words:
        cleaned = cleaned.replace(word, '')
    # Clean whitespace
    cleaned = ' '.join(cleaned.split())
    return cleaned


def get_base_ingredient(ingredient_name):
    """
    Extract base ingredient from specific variants
    
    Examples:
        "Chicken drumstick" → "chicken"
        "Beef ribeye" → "beef"
        "Salmon fillet" → "salmon"
    """
    ingredient_lower = ingredient_name.lower()
    
    # Base ingredient mapping (keyword → base)
    base_mapping = {
        # Poultry
        'chicken': 'chicken',
        'turkey': 'turkey',
        'duck': 'duck',
        'quail': 'quail',
        'goose': 'goose',
        
        # Meat
        'beef': 'beef',
        'pork': 'pork',
        'lamb': 'lamb',
        'veal': 'veal',
        
        # Seafood
        'salmon': 'salmon',
        'tuna': 'tuna',
        'cod': 'cod',
        'shrimp': 'shrimp',
        'prawn': 'shrimp',  # Map prawn to shrimp
        'crab': 'crab',
        'lobster': 'lobster',
        'fish': 'fish',
        
        # Vegetables
        'potato': 'potato',
        'tomato': 'tomato',
        'carrot': 'carrot',
        'onion': 'onion',
        'pepper': 'bell pepper',
        'broccoli': 'broccoli',
        'spinach': 'spinach',
        'lettuce': 'lettuce',
        'cabbage': 'cabbage',
        
        # Fruits
        'apple': 'apple',
        'banana': 'banana',
        'orange': 'orange',
        'grape': 'grape',
        'strawberry': 'strawberry',
        
        # Grains/Legumes
        'rice': 'rice',
        'wheat': 'wheat',
        'oat': 'oats',
        'bean': 'beans',
        
        # Dairy
        'milk': 'milk',
        'cheese': 'cheese',
        'yogurt': 'yogurt',
        'egg': 'egg',
    }
    
    # Find matching base ingredient
    for keyword, base in base_mapping.items():
        if keyword in ingredient_lower:
            return base
    
    # If no match, return first word
    return ingredient_name.split()[0].lower()


def search_usda(ingredient_name, df, prefer_raw=True):
    """
    Search USDA database for ingredient
    
    Args:
        ingredient_name: Ingredient to search
        df: USDA dataframe
        prefer_raw: Prefer raw/unprocessed foods
    
    Returns:
        dict with nutrition data or None
    """
    cleaned = clean_ingredient_name(ingredient_name)
    
    # Search patterns (in order of preference)
    search_terms = [
        cleaned,  # Full cleaned name
        cleaned.split()[0] if ' ' in cleaned else cleaned,  # First word (e.g., "chicken")
    ]
    
    for term in search_terms:
        # Search in description
        mask = df['description'].str.lower().str.contains(term, na=False, regex=False)
        results = df[mask].copy()
        
        if len(results) == 0:
            continue
        
        # Prefer raw/unprocessed
        if prefer_raw:
            raw_mask = results['description'].str.lower().str.contains(
                'raw|fresh|uncooked',
                na=False,
                regex=True
            )
            if raw_mask.any():
                results = results[raw_mask]
        
        # Prioritize data types
        data_type_priority = ['sr_legacy_food', 'foundation_food', 'survey_fndds_food', 'branded_food']
        
        for data_type in data_type_priority:
            type_results = results[results['data_type'] == data_type]
            if len(type_results) > 0:
                # Take first result
                best = type_results.iloc[0]
                
                return {
                    'ingredient': ingredient_name,
                    'fdc_id': int(best['fdc_id']),
                    'description': best['description'],
                    'data_type': best['data_type'],
                    'calories': float(best['calories']) if pd.notna(best['calories']) else 0.0,
                    'protein_g': float(best.get('protein_g', 0)) if pd.notna(best.get('protein_g')) else 0.0,
                    'fat_g': float(best.get('fat_g', 0)) if pd.notna(best.get('fat_g')) else 0.0,
                    'carbs_g': float(best.get('carbs_g', 0)) if pd.notna(best.get('carbs_g')) else 0.0,
                    'search_term': term,
                    'match_quality': 'exact' if term == cleaned else 'partial'
                }
    
    # Not found
    return None


print("✓ Search functions defined")
print("✓ Mapping system ready")

✓ Search functions defined
✓ Mapping system ready


## 4. Ingredient Mapping System

**For ingredients not found in USDA**, automatically map to base ingredients:
- "Chicken drumstick" → "Chicken breast"
- "Beef tenderloin" → "Beef"  
- "Salmon belly" → "Salmon fillet"

In [6]:
# Process all ingredients
nutrition_data = []
not_found = []
mapped_count = 0

# First pass: collect all found ingredients
found_lookup = {}

print(f"Pass 1: Searching USDA for {len(ingredients_list)} ingredients...\n")

for ingredient in tqdm(ingredients_list, desc="Searching USDA"):
    result = search_usda(ingredient, df_usda)
    
    if result:
        nutrition_data.append(result)
        # Store for mapping lookup
        base = get_base_ingredient(ingredient)
        if base not in found_lookup:
            found_lookup[base] = result
    else:
        not_found.append(ingredient)

print(f"\n✓ Pass 1 complete")
print(f"  - Found in USDA: {len(nutrition_data)}")
print(f"  - Not found: {len(not_found)}")

# Second pass: Map not found ingredients to base ingredients
if not_found:
    print(f"\nPass 2: Mapping {len(not_found)} ingredients to base types...\n")
    
    for ingredient in tqdm(not_found, desc="Mapping"):
        base = get_base_ingredient(ingredient)
        
        # Try to find base ingredient in our found data
        if base in found_lookup:
            base_data = found_lookup[base]
            nutrition_data.append({
                'ingredient': ingredient,
                'fdc_id': base_data['fdc_id'],
                'description': f"MAPPED: {base_data['description']}",
                'data_type': 'mapped',
                'calories': base_data['calories'],
                'protein_g': base_data['protein_g'],
                'fat_g': base_data['fat_g'],
                'carbs_g': base_data['carbs_g'],
                'search_term': base,
                'match_quality': 'mapped'
            })
            mapped_count += 1
        else:
            # Still not found, use zeros
            nutrition_data.append({
                'ingredient': ingredient,
                'fdc_id': 0,
                'description': f'NOT FOUND: {ingredient}',
                'data_type': 'not_found',
                'calories': 0.0,
                'protein_g': 0.0,
                'fat_g': 0.0,
                'carbs_g': 0.0,
                'search_term': '',
                'match_quality': 'not_found'
            })

print(f"\n{'='*80}")
print(f"PROCESSING COMPLETE")
print(f"{'='*80}")
print(f"Total ingredients: {len(ingredients_list)}")
print(f"  ✓ Found in USDA: {len(nutrition_data) - mapped_count - (len(not_found) - mapped_count)}")
print(f"  ✓ Mapped to base: {mapped_count}")
print(f"  ✗ Still not found: {len(not_found) - mapped_count}")
print(f"\nCoverage: {((len(nutrition_data) - (len(not_found) - mapped_count)) / len(ingredients_list) * 100):.1f}%")

Pass 1: Searching USDA for 528 ingredients...



Searching USDA: 100%|██████████| 528/528 [04:21<00:00,  2.02it/s]



✓ Pass 1 complete
  - Found in USDA: 524
  - Not found: 4

Pass 2: Mapping 4 ingredients to base types...



Mapping: 100%|██████████| 4/4 [00:00<?, ?it/s]


PROCESSING COMPLETE
Total ingredients: 528
  ✓ Found in USDA: 524
  ✓ Mapped to base: 0
  ✗ Still not found: 4

Coverage: 99.2%


## 5. Review Results

In [7]:
# Create dataframe
df_nutrition = pd.DataFrame(nutrition_data)

print(f"✓ Nutrition database created: {len(df_nutrition)} ingredients")
print(f"\nData type distribution:")
print(df_nutrition['data_type'].value_counts())
print(f"\nMatch quality:")
print(df_nutrition['match_quality'].value_counts())
print(f"\nSample data:")
display(df_nutrition.head(20))

✓ Nutrition database created: 528 ingredients

Data type distribution:
data_type
branded_food         268
sr_legacy_food       231
survey_fndds_food     24
not_found              4
foundation_food        1
Name: count, dtype: int64

Match quality:
match_quality
exact        497
partial       27
not_found      4
Name: count, dtype: int64

Sample data:


,ingredient,fdc_id,description,data_type,calories,protein_g,fat_g,carbs_g,search_term,match_quality
0,Chicken breast,171515,"Chicken breast tenders, breaded, uncooked",sr_legacy_food,263.0,14.73,15.75,15.01,chicken breast,exact
1,Chicken thigh,2706029,"Chicken thigh, baked, broiled, or roasted, ski...",survey_fndds_food,231.0,23.12,14.62,0.00,chicken thigh,exact
2,Chicken drumstick,2706002,"Chicken drumstick, baked, broiled, or roasted,...",survey_fndds_food,190.0,23.21,10.09,0.00,chicken drumstick,exact
3,Chicken wing,2706057,"Chicken wing, baked, broiled, or roasted, from...",survey_fndds_food,252.0,23.60,16.74,0.00,chicken wing,exact
4,Chicken liver,360612,"SANDERSON FARMS, FRESH CHICKEN LIVERS",branded_food,116.0,16.96,4.46,0.89,chicken liver,exact
5,Chicken gizzard,2719261,Tyson All Natural* Fresh Chicken Gizzards & He...,branded_food,69.0,13.93,0.91,0.00,chicken gizzard,exact
6,Chicken feet,2706085,Chicken feet,survey_fndds_food,215.0,19.40,14.60,0.20,chicken feet,exact
7,Ground chicken,2417777,96% LEAN 4% FAT FRESH GROUND CHICKEN,branded_food,107.0,21.43,3.12,0.00,ground chicken,exact
8,Turkey breast,808662,3% Fresh Young Bone-In Turkey Breast 3-8 lbs.,branded_food,161.0,20.54,8.04,0.00,turkey breast,exact
9,Turkey thigh,2649714,"Turkey Thighs, Bone-in, 3 Count Tray, Fresh, 1...",branded_food,195.0,17.00,8.00,0.00,turkey thigh,exact


In [8]:
# Statistics (excluding not found)
df_found = df_nutrition[df_nutrition['data_type'] != 'not_found']

print(f"Nutrition statistics (found items only):")
print(df_found[['calories', 'protein_g', 'fat_g', 'carbs_g']].describe())

Nutrition statistics (found items only):
          calories   protein_g       fat_g     carbs_g
count   524.000000  524.000000  524.000000  524.000000
mean    220.083969    9.077615   10.866508   22.427004
std     223.374238   10.906857   21.735062   28.576604
min       0.000000    0.000000    0.000000    0.000000
25%      62.750000    1.492500    0.250000    2.695000
50%     159.500000    5.050000    2.060000   10.870000
75%     339.500000   14.290000   13.330000   33.835000
max    2867.000000   88.890000  273.330000  300.000000


## 6. Save Results

In [9]:
# Save CSV
df_nutrition.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
print(f"✓ Saved CSV: {OUTPUT_CSV}")
print(f"  Size: {OUTPUT_CSV.stat().st_size / 1024:.1f} KB")

✓ Saved CSV: c:\Users\Champion\Documents\GitHub\cAIuldron\data\ingredients_nutrition_full.csv
  Size: 64.2 KB


In [10]:
# Save JSON lookup (for fast access in code)
nutrition_lookup = {}

for _, row in df_nutrition.iterrows():
    nutrition_lookup[row['ingredient']] = {
        'calories': row['calories'],
        'protein_g': row['protein_g'],
        'fat_g': row['fat_g'],
        'carbs_g': row['carbs_g'],
        'description': row['description'],
        'data_type': row['data_type']
    }

with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(nutrition_lookup, f, indent=2, ensure_ascii=False)

print(f"✓ Saved JSON: {OUTPUT_JSON}")
print(f"  Size: {OUTPUT_JSON.stat().st_size / 1024:.1f} KB")

✓ Saved JSON: c:\Users\Champion\Documents\GitHub\cAIuldron\data\nutrition_lookup_full.json
  Size: 114.9 KB


## 7. Usage Example

In [11]:
# Test usage
def get_nutrition(ingredient, weight_g=100):
    """Get nutrition for ingredient at given weight"""
    if ingredient in nutrition_lookup:
        data = nutrition_lookup[ingredient]
        multiplier = weight_g / 100
        
        return {
            'ingredient': ingredient,
            'weight_g': weight_g,
            'calories': data['calories'] * multiplier,
            'protein_g': data['protein_g'] * multiplier,
            'fat_g': data['fat_g'] * multiplier,
            'carbs_g': data['carbs_g'] * multiplier
        }
    return None

# Test with some ingredients
test_ingredients = ['Chicken breast', 'Salmon fillet', 'Broccoli', 'Rice']

print("Test queries:\n")
for ing in test_ingredients:
    result = get_nutrition(ing, 150)
    if result:
        print(f"{ing} (150g):")
        print(f"  Calories: {result['calories']:.1f} kcal")
        print(f"  Protein: {result['protein_g']:.1f}g")
        print(f"  Fat: {result['fat_g']:.1f}g")
        print(f"  Carbs: {result['carbs_g']:.1f}g")
        print()
    else:
        print(f"{ing}: NOT FOUND\n")

Test queries:

Chicken breast (150g):
  Calories: 394.5 kcal
  Protein: 22.1g
  Fat: 23.6g
  Carbs: 22.5g

Salmon fillet (150g):
  Calories: 325.5 kcal
  Protein: 30.0g
  Fat: 20.0g
  Carbs: 3.8g

Broccoli (150g):
  Calories: 33.0 kcal
  Protein: 4.8g
  Fat: 0.7g
  Carbs: 4.3g

Rice (150g):
  Calories: 550.5 kcal
  Protein: 11.3g
  Fat: 4.8g
  Carbs: 114.4g



## 8. Summary

### Output Files:
1. **`data/ingredients_nutrition_full.csv`** - Complete nutrition table with all 529 ingredients
2. **`data/nutrition_lookup_full.json`** - JSON lookup for fast access

### Coverage:
- **529 total ingredients**
- **Found in USDA**: Will be shown in results above
- **Not found**: Listed above (placeholders with 0 values)

### Next Steps:
1. Review ingredients not found in USDA
2. Manually add nutrition data for missing items if needed
3. Integrate into pipeline using the JSON lookup file

### Usage in Pipeline:
```python
import json

# Load lookup table
with open('data/nutrition_lookup_full.json') as f:
    NUTRITION_DB = json.load(f)

# Get nutrition
ingredient = 'Chicken breast'
if ingredient in NUTRITION_DB:
    nutrition = NUTRITION_DB[ingredient]
    print(f"Calories: {nutrition['calories']} per 100g")
```

In [ ]:
print("="*80)
print("NUTRITION DATABASE GENERATION COMPLETE")
print("="*80)
print(f"\n✓ Total ingredients processed: {len(df_nutrition)}")
print(f"✓ Found in USDA: {len(df_found)}")
print(f"✓ Not found: {len(not_found)}")
print(f"\nOutput files:")
print(f"  - {OUTPUT_CSV}")
print(f"  - {OUTPUT_JSON}")
print("\nReady to integrate into pipeline!")
print("="*80)